In [0]:
# ---------------------------------------------------------
# 0. Run Imports 
# ---------------------------------------------------------

import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
# ---------------------------------------------------------
# 1. Extract Function to Write Ingestion Observability Data
# ---------------------------------------------------------
def write_ingestion_observability(
    spark,
    observability_table,
    source_system,
    bronze_table,
    batch_id,
    pipeline_run_id,
    file_path,
    file_size,
    file_hash,
    ingestion_start_timestamp,
    ingestion_end_timestamp,
    total_input_rows,
    valid_rows,
    schema_change_count,
    ingestion_status,
    error_message=None
):

    duration_seconds = (
        ingestion_end_timestamp
        - ingestion_start_timestamp
    ).total_seconds()

    rows = [(
        source_system,
        bronze_table,
        batch_id,
        pipeline_run_id,
        file_path,
        file_path.split("/")[-1],
        file_size,
        file_hash,
        ingestion_start_timestamp,
        ingestion_end_timestamp,
        duration_seconds,
        total_input_rows,
        valid_rows,
        schema_change_count,
        ingestion_status,
        error_message
    )]

    # Define explicit schema to avoid type inference issues
    schema = StructType([
        StructField("source_system", StringType(), True),
        StructField("bronze_table", StringType(), True),
        StructField("batch_id", StringType(), True),
        StructField("pipeline_run_id", StringType(), True),
        StructField("file_path", StringType(), True),
        StructField("file_name", StringType(), True),
        StructField("file_size", LongType(), True),
        StructField("file_hash", StringType(), True),
        StructField("ingestion_start_timestamp", TimestampType(), True),
        StructField("ingestion_end_timestamp", TimestampType(), True),
        StructField("ingestion_duration_seconds", DoubleType(), True),
        StructField("total_input_rows", LongType(), True),
        StructField("valid_rows", LongType(), True),
        StructField("schema_change_count", LongType(), True),
        StructField("ingestion_status", StringType(), True),
        StructField("error_message", StringType(), True)
    ])

    observability_df = spark.createDataFrame(
        rows,
        schema
    ).withColumn(
        "created_timestamp",
        F.current_timestamp()
    )

    (
        observability_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(observability_table)
    )